# Step 3 walkthrough: how `labels_v1` is built, one piece at a time

This notebook runs the **real** step 3 code (`src/cropyield/data/labels.py`) on the **real** data, but in small cells.
Each cell keeps its result in a variable, so you can `print()` anything.

**Rules of this notebook**
- It reads from `C:\Dev\DSC-291-Final-Project`. It **writes nothing** there.
- Where a function is copied into a cell (so you can print inside it), a check cell proves that the copy gives
  exactly the same result as the real function. If you change a copied line, that check will tell you.
- Kernel: choose the Python of `C:\Dev\DSC-291-Final-Project\.venv`.
- Run All once (about 1–2 minutes: the OAF workbook takes about a minute). Then add your prints and rerun single cells.

**The path, and the variable that holds each stage**

```text
Stage A  lookups        lacuna, oaf            raw columns the audit dropped (Lacuna, Kenya only)
Stage B  one cohort     rows -> frame          cohort_frame() unrolled: shared block, obs_date, if-blocks
Stage C  cohort flags   frame                  add_cohort_flags() unrolled: clusters, duplicates, dates
Stage D  all 5 cohorts  frames -> labels       the real functions in a loop, then pd.concat
Stage E  inclusion      labels                 exclusion_reasons() unrolled, include_primary, column order
Stage F  checks         problems, frozen       the 11 checks, and labels == the frozen labels_v1.parquet
```

In [1]:
import sys
from pathlib import Path

REPO = Path(r"C:\Dev\DSC-291-Final-Project")
sys.path.insert(0, str(REPO / "src"))

import numpy as np
import pandas as pd

import cropyield
from cropyield.data import labels as L          # the real step 3 module
from cropyield.data.cohorts import COHORTS, load_cohort

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
print("code is read from:", cropyield.__file__)
print("the 5 cohorts, in build order:", list(COHORTS))

code is read from: C:\Dev\DSC-291-Final-Project\src\cropyield\__init__.py
the 5 cohorts, in build order: ['malawi_2022', 'malawi_2023', 'kenya_oaf_2019_2020', 'zimbabwe_lacuna_2022_2023', 'zimbabwe_2024']


In [2]:
# The field we follow through every stage. Lacuna source_row 474 (the row we traced before).
LACUNA_ROW = "zimbabwe:c12679fc-cb3e-4fb8-9bf3-ccf055c05640"
FOLLOW = LACUNA_ROW


def show_row(table, title, obs_id=None):
    '''Print the size of a table and every column of the followed row, one per line.'''
    obs_id = obs_id or FOLLOW
    print(f"--- {title}: {len(table):,} rows x {table.shape[1]} columns")
    if "obs_id" not in table:
        print("    (this table has no obs_id column)")
        return
    hit = table[table["obs_id"] == obs_id]
    if hit.empty:
        print(f"    (the followed row {obs_id} is not in this table)")
        return
    for column, value in hit.iloc[0].items():
        print(f"    {column:30s} {value!r}")

## Stage A. Two lookups: raw columns the audit dropped

These are **not cohorts**. Step 2 (the audit) did not keep some raw columns that we need:
- Kenya: `district`, `site`, the site coordinates, `plot_acres` → `L.oaf_groups()` reads the raw Excel workbook again.
- Lacuna: `Admin1`, `Field_Irri` → `L.lacuna_groups()` reads the raw GeoJSON again.

Malawi and Zimbabwe 2024 need no lookup: what they need is already in the audit files.
Each lookup also keeps the raw yield and coordinates, so that Stage F can compare them with the table.

In [3]:
lacuna = L.lacuna_groups()            # key: source_id
print(lacuna.shape)
print(lacuna[lacuna["source_id"] == FOLLOW.split(":", 1)[1]].T)   # our field in the lookup
lacuna.head()

(2269, 7)
                                                 169
source_id       c12679fc-cb3e-4fb8-9bf3-ccf055c05640
Admin1                                     Marondera
Field_Irri                                        no
raw_yield_t_ha                                   8.2
raw_dry_kg                                      32.8
raw_lon                                    31.194411
raw_lat                                   -18.490017


,source_id,Admin1,Field_Irri,raw_yield_t_ha,raw_dry_kg,raw_lon,raw_lat
0,6e287d1e-329e-48f5-8564-3683c02772c5,Mvurwi,no,0.659375,2.6375,30.892344,-17.281249
1,9c82453b-e849-4480-b2a7-fe2dd5001df2,Mvurwi,no,3.170000,12.6800,30.886695,-17.169705
2,bf3a7322-276d-4ae9-ab65-68463e751c19,Bindura,no,3.575000,14.3000,31.367041,-17.512770
3,97977edf-8123-4a0f-b315-9c9456bb83f9,Mvurwi,no,3.760000,15.0400,30.898790,-17.309835
4,a8c25796-f2df-4a5b-a6b3-38623b49f9c7,Mvurwi,no,2.188750,8.7550,30.886618,-17.170793


In [4]:
oaf = L.oaf_groups()                  # key: source_row (row number in the workbook). Takes about a minute.
print(oaf.shape)
oaf.head()

(23674, 11)


,source_row,unique_id,district,site,site_latitude,site_longitude,plot_acres,yield_kg_ph,yield_kg_pa,field_latitude,field_longitude
0,2,b801f443,aldai,Busali,NaN,NaN,0.03,2965.2600,1200.00000,0.135177,34.803463
1,3,404a949c,aldai,Busali,NaN,NaN,0.10,2340.9951,947.36841,0.127124,34.803947
2,4,39d6d352,aldai,Busali,NaN,NaN,0.25,4103.2432,1660.52600,0.137745,34.824932
3,5,5faab0e4,aldai,Busali,NaN,NaN,0.25,4187.7798,1694.73710,0.125212,34.815304
4,6,50032d92,aldai,Busali,NaN,NaN,0.25,5000.6250,2023.68400,0.135209,34.803474


## Stage B. One cohort by hand: `cohort_frame()` unrolled

Choose a cohort in the next cell. The cells after it are a **copy** of `cohort_frame()` (`labels.py` lines 169–219),
cut into its three parts. Cell B4 proves that the copy equals the real function.

**The one trick that makes this code look strange.** In `pd.DataFrame({...})`, a column can be a whole column
(`rows["yield_kg_ha"]`) or **one single value** (`m.role`, `np.nan`, `None`). pandas copies a single value onto
**every row**. So `"cut_area_m2": float(m.cut_area_m2)` writes 40.0 on all 2,266 Lacuna rows. That is how the
cohort constants (cut size, number of cuts, moisture) get into the table, and how the empty placeholders are made.

In [5]:
COHORT = "zimbabwe_lacuna_2022_2023"
# Try also: "malawi_2023", "kenya_oaf_2019_2020", "zimbabwe_2024", "malawi_2022"

rows = load_cohort(COHORT)            # step 2 output (screened audit CSV) + obs_id + season window
m = L.MEASUREMENTS[COHORT]            # the constants of this cohort
name = COHORT                         # the real function calls it `name`
FOLLOW = LACUNA_ROW if COHORT == "zimbabwe_lacuna_2022_2023" else rows["obs_id"].iloc[0]

print(m)
show_row(rows, "B0  rows = load_cohort(COHORT)")

Measurement(role='second', iso3='ZWE', definition='cut_8x5m_dry_weight', cut_area_m2=40, n_cuts=1, moisture_pct=None, unit_provisional=True, location_source='field_gps', source_crs='OGC:CRS84')
--- B0  rows = load_cohort(COHORT): 2,266 rows x 35 columns
    obs_id                         'zimbabwe:c12679fc-cb3e-4fb8-9bf3-ccf055c05640'
    dataset                        'zimbabwe'
    source_row                     np.int64(474)
    source_id                      'c12679fc-cb3e-4fb8-9bf3-ccf055c05640'
    country                        'zimbabwe'
    year                           np.int64(2023)
    season                         ''
    crop                           'maize'
    yield_field                    'Yield(Mt/H'
    source_yield                   '8.2'
    source_yield_units             't/ha'
    yield_kg_ha                    np.float64(8200.0)
    longitude                      np.float64(31.19441122115361)
    latitude                       np.float64(-18.490016844567556)


In [6]:
# B1. The shared block: the same columns for every cohort (labels.py lines 172-190).
frame = pd.DataFrame({
    "obs_id": rows["obs_id"], "cohort": name, "role": m.role, "dataset": rows["dataset"],
    "source_id": rows["source_id"], "source_row": rows["source_row"],
    "source_sha256": L.sha256(L.AUDIT_DIR / f"{COHORTS[name].dataset}_screened_candidates.csv"),
    "country_iso3": m.iso3, "season_year": rows["year"],
    "season_start": pd.to_datetime(rows["season_start"]), "season_end": pd.to_datetime(rows["season_end"]),
    "yield_kg_ha": rows["yield_kg_ha"],
    "yield_definition": m.definition, "cut_area_m2": float(m.cut_area_m2), "n_cuts": m.n_cuts,
    "moisture_basis_pct": m.moisture_pct, "unit_provisional": m.unit_provisional,
    "cut1_kg_ha": np.nan, "cut2_kg_ha": np.nan,                                   # placeholders
    "geometry_type": rows["geometry_type"], "lat": rows["latitude"], "lon": rows["longitude"],
    "field_area_ha": L.number_column(rows, "field_area_ha"), "location_source": m.location_source,
    "source_crs": m.source_crs,
    "plant_date": L.dates(L.text_column(rows, "planting_date")),
    "harvest_date": L.dates(L.text_column(rows, "harvest_date")),
    "intercropped": L.text_column(rows, "intercropping").map({"0": False, "no": False, "1": True, "yes": True}),
    "irrigated": pd.NA, "admin_unit": None, "site_unit": None, "site_distance_km": np.nan,   # placeholders
    "source_flags": rows["quality_flags"],
})
frame["intercropped"] = frame["intercropped"].astype("boolean")
frame["irrigated"] = frame["irrigated"].astype("boolean")

show_row(frame, "B1  after the shared block")

--- B1  after the shared block: 2,266 rows x 33 columns
    obs_id                         'zimbabwe:c12679fc-cb3e-4fb8-9bf3-ccf055c05640'
    cohort                         'zimbabwe_lacuna_2022_2023'
    role                           'second'
    dataset                        'zimbabwe'
    source_id                      'c12679fc-cb3e-4fb8-9bf3-ccf055c05640'
    source_row                     np.int64(474)
    source_sha256                  'b5760d3ea75d3019d06c97f8f77ad4d32b5a796185537fc940fc6aab3bd4d14c'
    country_iso3                   'ZWE'
    season_year                    np.int64(2023)
    season_start                   Timestamp('2022-10-01 00:00:00')
    season_end                     Timestamp('2023-06-30 00:00:00')
    yield_kg_ha                    np.float64(8200.0)
    yield_definition               'cut_8x5m_dry_weight'
    cut_area_m2                    np.float64(40.0)
    n_cuts                         np.int64(1)
    moisture_basis_pct             None
    un

In [7]:
# B2. obs_date: the harvest date, else the crop-cut or photo date, else empty (labels.py lines 191-196).
observed = L.dates(L.text_column(rows, "observation_time"))
frame["obs_date"] = frame["harvest_date"].fillna(observed)
frame["obs_date_source"] = np.where(frame["harvest_date"].notna(), "harvest_date", np.where(
    observed.notna(), "photo_filename" if name == "zimbabwe_2024" else "crop_cut_time", "none"))
frame["geometry_wkt"] = None

print(frame["obs_date_source"].value_counts())

obs_date_source
harvest_date    2266
Name: count, dtype: int64


In [8]:
# B3. The if-blocks: only the cohort that has the data fills its placeholders (labels.py lines 197-218).
ran = []
if name.startswith("malawi"):
    ran.append("malawi: polygon from the audit GeoJSON")
    polygons = L.field_polygons(COHORTS[name].dataset)
    frame["geometry_wkt"] = frame["source_id"].map(polygons)
if name == "malawi_2023":
    ran.append("malawi_2023: main cut and nested cut")
    frame["cut1_kg_ha"] = rows["yield_kg_ha"]
    frame["cut2_kg_ha"] = L.number_column(rows, "nested_subcut_yield_t_ha") * 1000
if name == "zimbabwe_2024":
    ran.append("zimbabwe_2024: box 1 and box 2")
    frame["cut1_kg_ha"] = L.number_column(rows, "box1_yield_t_ha") * 1000
    frame["cut2_kg_ha"] = L.number_column(rows, "box2_yield_t_ha") * 1000
if name == "kenya_oaf_2019_2020":
    ran.append("kenya: join the oaf lookup on source_row")
    raw = rows[["source_row"]].merge(oaf, on="source_row", how="left", validate="one_to_one")
    frame["admin_unit"] = L.normal_district(raw["district"].fillna("").astype(str)).to_numpy()
    site = raw["site"].fillna("").astype(str).str.lower().str.strip()
    frame["site_unit"] = (frame["admin_unit"] + "|" + site.to_numpy()).to_numpy()
    site_lat = pd.to_numeric(raw["site_latitude"], errors="coerce").to_numpy()
    site_lon = pd.to_numeric(raw["site_longitude"], errors="coerce").to_numpy()
    frame["site_distance_km"] = L.geodesic_km(frame["lat"].to_numpy(), frame["lon"].to_numpy(), site_lat, site_lon)
    frame["field_area_ha"] = (pd.to_numeric(raw["plot_acres"], errors="coerce") * L.ACRE_HA).to_numpy()
if name == "zimbabwe_lacuna_2022_2023":
    ran.append("lacuna: join the lacuna lookup on source_id")
    raw = rows[["source_id"]].merge(lacuna, on="source_id", how="left", validate="one_to_one")
    frame["admin_unit"] = raw["Admin1"].str.strip().str.lower().to_numpy()
    frame["irrigated"] = raw["Field_Irri"].map({"no": False, "yes": True}).astype("boolean").to_numpy()

print("if-blocks that ran for", name, "->", ran)
show_row(frame, "B3  after the if-blocks")

if-blocks that ran for

 zimbabwe_lacuna_2022_2023 -> ['lacuna: join the lacuna lookup on source_id']
--- B3  after the if-blocks: 2,266 rows x 36 columns
    obs_id                         'zimbabwe:c12679fc-cb3e-4fb8-9bf3-ccf055c05640'
    cohort                         'zimbabwe_lacuna_2022_2023'
    role                           'second'
    dataset                        'zimbabwe'
    source_id                      'c12679fc-cb3e-4fb8-9bf3-ccf055c05640'
    source_row                     np.int64(474)
    source_sha256                  'b5760d3ea75d3019d06c97f8f77ad4d32b5a796185537fc940fc6aab3bd4d14c'
    country_iso3                   'ZWE'
    season_year                    np.int64(2023)
    season_start                   Timestamp('2022-10-01 00:00:00')
    season_end                     Timestamp('2023-06-30 00:00:00')
    yield_kg_ha                    np.float64(8200.0)
    yield_definition               'cut_8x5m_dry_weight'
    cut_area_m2                    np.float64(40.0)
    n_cuts        

In [9]:
# B4. Check: the hand-made frame equals the real cohort_frame().
real_frame = L.cohort_frame(name, oaf, lacuna)
pd.testing.assert_frame_equal(frame, real_frame)
print(f"OK: the copy above equals L.cohort_frame({name!r}) on all {len(frame):,} rows and {frame.shape[1]} columns.")

OK: the copy above equals L.cohort_frame('zimbabwe_lacuna_2022_2023') on all 2,266 rows and 36 columns.


## Stage C. Flags that compare rows: `add_cohort_flags()` unrolled

These flags need **all rows of one cohort at once**, so they run per cohort (`labels.py` lines 222–237).
That is why `cluster_2km` numbers start again at 0 in every cohort.

In [10]:
flagged = frame.copy()
lat, lon = flagged["lat"].to_numpy(), flagged["lon"].to_numpy()
flagged["cluster_2km"] = L.connected_clusters(lat, lon, 2.0)        # chains of fields with gaps under 2 km
flagged["near_dup_10m"] = L.has_neighbour(lat, lon, 0.01)           # another field within 10 m
flagged["polygon_overlap_dup"] = False
if flagged["geometry_wkt"].notna().all():                           # only Malawi has polygons
    flagged["polygon_overlap_dup"] = L.overlapping_polygons(flagged["geometry_wkt"])
flagged["is_zero"] = flagged["yield_kg_ha"] == 0
flagged["date_outside_window"] = (flagged["obs_date"] < flagged["season_start"]) | (flagged["obs_date"] > flagged["season_end"])
year = flagged["season_year"]
plant, harvest = flagged["plant_date"], flagged["harvest_date"]
flagged["impossible_date"] = ((plant.notna() & ((plant.dt.year < year - 1) | (plant.dt.year > year)))
                              | (harvest.notna() & plant.notna() & (harvest <= plant))
                              | (harvest.notna() & ((harvest.dt.year < year) | (harvest.dt.year > year + 1))))

print("clusters:", flagged["cluster_2km"].nunique(), "| numbers run from", flagged["cluster_2km"].min(), "to", flagged["cluster_2km"].max())
for column in ("near_dup_10m", "polygon_overlap_dup", "is_zero", "date_outside_window", "impossible_date"):
    print(f"{column:22s} {int(flagged[column].sum()):>5} rows True")

pd.testing.assert_frame_equal(flagged, L.add_cohort_flags(real_frame.copy()))
print("OK: equals L.add_cohort_flags().")

clusters: 342 | numbers run from 0 to 341
near_dup_10m              28 rows True
polygon_overlap_dup        0 rows True
is_zero                   28 rows True
date_outside_window       22 rows True
impossible_date            0 rows True


OK: equals L.add_cohort_flags().


## Stage D. All five cohorts, then stack them

This is `build_labels()` (`labels.py` line 267), first half. It runs Stage B and Stage C for each cohort with the
real functions, then puts the five tables **on top of each other** with `pd.concat`. They can be stacked because
the shared block gave every cohort the same columns.

In [11]:
frames = []
for cohort_name in COHORTS:
    one = L.add_cohort_flags(L.cohort_frame(cohort_name, oaf, lacuna))
    print(f"{cohort_name:28s} {one.shape[0]:>5} rows x {one.shape[1]} columns")
    frames.append(one)

labels = pd.concat(frames, ignore_index=True)
print("\nstacked:", labels.shape)
for cohort_name, block in labels.groupby("cohort", sort=False):
    print(f"   {cohort_name:28s} rows {block.index.min():>5} .. {block.index.max():>5}")

malawi_2022                    529 rows x 42 columns


malawi_2023                    260 rows x 42 columns


kenya_oaf_2019_2020           3977 rows x 42 columns
zimbabwe_lacuna_2022_2023     2266 rows x 42 columns


zimbabwe_2024                 4332 rows x 42 columns

stacked: (11364, 42)
   malawi_2022                  rows     0 ..   528
   malawi_2023                  rows   529 ..   788
   kenya_oaf_2019_2020          rows   789 ..  4765
   zimbabwe_lacuna_2022_2023    rows  4766 ..  7031
   zimbabwe_2024                rows  7032 .. 11363


In [12]:
# This is what print(labels) shows at this point. pandas hides the middle rows and columns; use show_row() or .T for one row.
print(labels)

                                                  obs_id         cohort     role                         dataset                             source_id  source_row                                      source_sha256  \
0       malawi_2022:3ee4d3a6-3a6e-4dae-9046-66009b51d884    malawi_2022  dropped                     malawi_2022  3ee4d3a6-3a6e-4dae-9046-66009b51d884           1  5eda1742f125e5f22847c5e8baa03c02a00adfbb770f1b...   
1       malawi_2022:4f780999-86d8-42b6-bc2a-3f45a6c56a63    malawi_2022  dropped                     malawi_2022  4f780999-86d8-42b6-bc2a-3f45a6c56a63           2  5eda1742f125e5f22847c5e8baa03c02a00adfbb770f1b...   
2       malawi_2022:9d9a8fb6-2b21-4d32-b9de-43f0d00af118    malawi_2022  dropped                     malawi_2022  9d9a8fb6-2b21-4d32-b9de-43f0d00af118           3  5eda1742f125e5f22847c5e8baa03c02a00adfbb770f1b...   
3       malawi_2022:b5dc6d2d-bc97-4f19-a3d6-9986325aeaf2    malawi_2022  dropped                     malawi_2022  b5dc6d2d-bc97-4f19

## Stage E. Who is in: `exclusion_reasons()` unrolled

Second half of `build_labels()`. Each condition is a True/False column over **all 11,364 rows**.
The loop goes **backwards**, so a later condition is written first and an earlier one overwrites it:
the **first** condition in the list wins. A row with no reason is `include_primary = True`.

In [13]:
conditions = [   # labels.py lines 243-250
    (labels["cohort"] == "malawi_2022", "dropped_cohort_2022_imagery_gap"),
    ((labels["cohort"] == "zimbabwe_lacuna_2022_2023") & (labels["season_year"] == 2022), "dropped_year_2022_imagery_gap"),
    ((labels["cohort"] == "zimbabwe_lacuna_2022_2023") & labels["intercropped"].fillna(False).astype(bool), "lacuna_intercropped"),
    ((labels["cohort"] == "zimbabwe_lacuna_2022_2023") & labels["irrigated"].fillna(False).astype(bool), "lacuna_irrigated"),
    ((labels["cohort"] == "zimbabwe_2024") & labels["is_zero"], "zimbabwe_2024_zero_likely_not_measured"),
    (labels["yield_kg_ha"] > L.MAX_YIELD_KG_HA, "yield_above_15_t_ha"),
]
for condition, reason in conditions:
    print(f"{reason:40s} hits {int(condition.sum()):>5} rows")

reasons = pd.Series("", index=labels.index, dtype=object)
for condition, reason in reversed(conditions):
    reasons[condition] = reason

print("\nfinal reason (first one wins):")
print(reasons.replace("", "(included)").value_counts())

pd.testing.assert_series_equal(reasons, L.exclusion_reasons(labels))
print("OK: equals L.exclusion_reasons().")

dropped_cohort_2022_imagery_gap          hits   529 rows
dropped_year_2022_imagery_gap            hits   169 rows
lacuna_intercropped                      hits   225 rows
lacuna_irrigated                         hits     5 rows
zimbabwe_2024_zero_likely_not_measured   hits  1874 rows
yield_above_15_t_ha                      hits     0 rows

final reason (first one wins):
(included)                                8575
zimbabwe_2024_zero_likely_not_measured    1874
dropped_cohort_2022_imagery_gap            529
lacuna_intercropped                        212
dropped_year_2022_imagery_gap              169
lacuna_irrigated                             5
Name: count, dtype: int64
OK: equals L.exclusion_reasons().


In [14]:
labels["exclusion_reason"] = reasons
labels["include_primary"] = labels["exclusion_reason"] == ""
labels = labels[L.COLUMNS]            # fixed order of the 44 columns
print(labels.groupby("cohort", sort=False)["include_primary"].agg(rows="size", included="sum"))
show_row(labels, "E  final row")

                           rows  included
cohort                                   
malawi_2022                 529         0
malawi_2023                 260       260
kenya_oaf_2019_2020        3977      3977
zimbabwe_lacuna_2022_2023  2266      1880
zimbabwe_2024              4332      2458


--- E  final row: 11,364 rows x 44 columns
    obs_id                         'zimbabwe:c12679fc-cb3e-4fb8-9bf3-ccf055c05640'
    cohort                         'zimbabwe_lacuna_2022_2023'
    role                           'second'
    dataset                        'zimbabwe'
    source_id                      'c12679fc-cb3e-4fb8-9bf3-ccf055c05640'
    source_row                     np.int64(474)
    source_sha256                  'b5760d3ea75d3019d06c97f8f77ad4d32b5a796185537fc940fc6aab3bd4d14c'
    country_iso3                   'ZWE'
    season_year                    np.int64(2023)
    season_start                   Timestamp('2022-10-01 00:00:00')
    season_end                     Timestamp('2023-06-30 00:00:00')
    yield_kg_ha                    np.float64(8200.0)
    yield_definition               'cut_8x5m_dry_weight'
    cut_area_m2                    np.float64(40.0)
    n_cuts                         np.int64(1)
    moisture_basis_pct             None
    unit_provision

## Stage F. The checks, and the proof that this is the real `labels_v1`

`step3_build_labels.py` writes the parquet **only** if both check lists are empty. This notebook never writes it.
The last cell compares the table you built here with the frozen file on disk.

In [15]:
problems = L.check_labels(labels) + L.check_against_raw(labels, oaf, lacuna)
print("problems:", problems if problems else "none: all checks pass, so step 3 would write the file")

problems: none: all checks pass, so step 3 would write the file


In [16]:
# Step 3 writes labels.to_parquet(...). We do the same, but into memory, and read it back.
# (In memory an empty moisture value is None; in a parquet file it becomes NaN. The round trip makes them alike.)
import hashlib, io
buffer = io.BytesIO()
labels.to_parquet(buffer, index=False)
as_written = pd.read_parquet(io.BytesIO(buffer.getvalue()))

frozen = pd.read_parquet(L.LABELS_PATH)
print("frozen file:", L.LABELS_PATH, frozen.shape)
pd.testing.assert_frame_equal(as_written, frozen)
print("OK: what step 3 would write from this notebook equals the frozen labels_v1.parquet, cell for cell.")
print("same bytes too?", hashlib.sha256(buffer.getvalue()).hexdigest() == L.sha256(L.LABELS_PATH))

frozen file: C:\Dev\DSC-291-Final-Project\data\processed\labels_v1.parquet (11364, 44)
OK: what step 3 would write from this notebook equals the frozen labels_v1.parquet, cell for cell.
same bytes too? True


## Where to put your own `print()`

| You want to see | Put it in |
|---|---|
| what one cohort looks like before step 3 | cell B0: `print(rows.columns.tolist())`, `rows.head()` |
| how a constant reaches every row | cell B1: `print(frame[["cohort", "cut_area_m2", "n_cuts"]].head())` |
| what the Kenya join adds | set `COHORT = "kenya_oaf_2019_2020"`, rerun from B0, then in B3: `print(raw.head())` |
| which rows are near duplicates | cell C: `print(flagged[flagged["near_dup_10m"]][["obs_id", "lat", "lon", "yield_kg_ha"]])` |
| why one row is out | cell E: `show_row(labels, "why", obs_id="<an obs_id>")` |

If a check cell fails after you edit a copied line, your edit changed the result. That is the point: you can test
your idea of what a line does.